# gemini

In [ ]:
"""
LLM-as-a-Judge evaluation for Sinhala VQA using Gemini.

Scores each predicted answer 0, 0.25, 0.5, or 1:
  1    — correct / semantically equivalent
  0.5  — partially correct (multi-part answer, got some parts)
  0.25 — right category/type, wrong specific value (e.g. wrong colour, wrong direction)
  0    — wrong category entirely / random / nonsensical
"""

import os
import re
import json
import time
import base64
import copy
from pathlib import Path
from typing import Any

import google.generativeai as genai

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
# or set env GEMINI_API_KEY
MODEL_NAME     = "gemini-3-flash-preview"
DATASET_FOLDER = "../../benchmarking/data"
SLEEP_BETWEEN  = 0.3                         # seconds between API calls
VALID_SCORES   = (0, 0.25, 0.5, 1)

SYSTEM_INSTRUCTION = (
    "You are a strict Visual Question Answering evaluator. "
    "You will be shown an image and a question about it written in Sinhala. "
    "Use the image to verify the ground truth and judge the predicted answer — "  # add this
    "do not rely solely on the ground truth text. "   
    "Judge whether the predicted answer is correct using the ground truth as a semantic reference — "
    "not as the only valid wording. Treat semantically equivalent answers as correct "
    "(e.g. 'teddy', 'teddy bear', 'plushy toy' are all equivalent). "
    "Reply ONLY with a single JSON object: "
    "{\"score\": <0 or 0.25 or 0.5 or 1>, \"reason\": \"<one short sentence of max 15 words>\"}\n\n"
    "Score meanings:\n"
    "  1    — correct or semantically equivalent to the ground truth\n"
    "  0.5  — the ground truth requires multiple parts and the prediction covers some but not all "
    "(e.g. ground truth is 'red and blue', prediction is only 'blue')\n"
    "  0.25 — the answer type/category is correct but the specific value is wrong "
    "(e.g. question asks for a colour, ground truth is 'red', prediction is 'blue' — "
    "both are colours so the model understood what was asked, just got the wrong one; "
    "or question asks for a direction, ground truth is 'right', prediction is 'left')\n"
    "  0    — wrong category entirely, completely unrelated, or random/nonsensical text\n\n"
    "Important:\n"
    "- Use 0 when the prediction shows no understanding of what the question was asking for.\n"
    "- Use 0.25 when the model understood the question type but gave a wrong single value.\n"
    "- Use 0.5 when the answer is partially correct because the ground truth has multiple parts.\n"
    "- Use 1 only when the answer is fully correct or semantically equivalent."
)


# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────
def find_image(image_id: int, dataset_folder: str) -> str:
    extensions = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]
    for ext in extensions:
        image_path = os.path.join(dataset_folder, f"{image_id}{ext}")
        if os.path.exists(image_path):
            return image_path
        for root, _, _files in os.walk(dataset_folder):
            image_path = os.path.join(root, f"{image_id}{ext}")
            if os.path.exists(image_path):
                return image_path
    raise FileNotFoundError(f"Image {image_id} not found in {dataset_folder}")


def encode_image(image_path: str) -> tuple[str, str]:
    suffix = Path(image_path).suffix.lower()
    mime = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}.get(suffix, "image/jpeg")
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8"), mime


def build_prompt(question: str, ground_truth: str, predicted: str) -> str:
    return (
        f"Question (Sinhala): {question}\n"
        f"Ground truth answer: {ground_truth}\n"
        f"Predicted answer: {predicted}\n\n"
        "Score the predicted answer. Reply ONLY with JSON: "
        "{\"score\": <0 or 0.25 or 0.5 or 1>, \"reason\": \"...\"}"
    )


def parse_score(text: str) -> dict[str, Any]:
    """Extract score and reason — robust to truncated JSON."""
    text = (text or "").strip()

    # 1. Direct parse
    try:
        obj = json.loads(text)
        score = float(obj["score"])
        assert score in VALID_SCORES
        return {"score": score, "reason": obj.get("reason", "")}
    except Exception:
        pass

    # 2. Salvage first complete {...}
    l, r = text.find("{"), text.rfind("}")
    if 0 <= l < r:
        try:
            obj = json.loads(text[l:r+1])
            score = float(obj["score"])
            assert score in VALID_SCORES
            return {"score": score, "reason": obj.get("reason", "")}
        except Exception:
            pass

    # 3. Regex fallback — works even if JSON is truncated mid-reason
    m = re.search(r'"score"\s*:\s*(0\.25|0\.5|0|1)', text)
    if m:
        score = float(m.group(1))
        r_match = re.search(r'"reason"\s*:\s*"([^"]*)"?', text)
        reason = r_match.group(1) if r_match else "(reason truncated)"
        return {"score": score, "reason": reason}

    return {"score": None, "reason": f"parse_failed: {text[:120]}"}


def print_summary(all_scores: list[float]) -> None:
    if not all_scores:
        return
    avg = sum(all_scores) / len(all_scores)
    counts = {0: 0, 0.25: 0, 0.5: 0, 1: 0}
    for s in all_scores:
        if s in counts:
            counts[s] += 1
    n = len(all_scores)
    print(f"\nScore distribution over {n} QAs:")
    print(f"  1    (correct)             : {counts[1]:5d}  ({100*counts[1]/n:.1f}%)")
    print(f"  0.5  (partial multi-part)  : {counts[0.5]:5d}  ({100*counts[0.5]/n:.1f}%)")
    print(f"  0.25 (right type/wrong val): {counts[0.25]:5d}  ({100*counts[0.25]/n:.1f}%)")
    print(f"  0    (wrong)               : {counts[0]:5d}  ({100*counts[0]/n:.1f}%)")
    print(f"  Average score              : {avg:.4f}")


def _save(out_items, wrap_key, original_data, out_path: Path):
    if wrap_key:
        out = {**original_data, wrap_key: out_items}
    else:
        out = out_items
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    api_key = API_KEY or os.environ.get("GEMINI_API_KEY", "")
    if not api_key:
        raise RuntimeError("Set API_KEY or export GEMINI_API_KEY.")

    genai.configure(api_key=api_key)
    gemini = genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=SYSTEM_INSTRUCTION,
    )

    # files = os.listdir("../bench-results-new")  # uncomment to run all
    files = [
            "benchmarking-results-qwen3-vl-2b-instruct.json",
        

        # "gemma-3-4b-si-cpt-mixed-vqa-38k-prompted.json",
        # "gemma-3-4b-si-cpt-mixed-vqa-38k.json",
        # "gemma-3-4b-si-vqa-38k.json",
        # "gemma-3-4b-si-cpt-vqa-15k.json",
        # "gemma-3-4b-si-vqa-15k.json",
        # "gemma-3-4b-si-cpt-vqa-38k.json",
        
        # "gemma-3-4b-si-vqa-5k.json"
            ]
    # files to skip even if present in the list/folder
    SKIP = {"scores.json"}

    for file in files:
        if file in SKIP:
            continue

        IN_PATH  = Path(f"../1_GEMINI/{file}")
        OUT_PATH = Path(f"{file}")
        print(f"\n{'='*60}")
        print(f"Processing: {file}")
        print(f"{'='*60}")

        data = json.loads(IN_PATH.read_text(encoding="utf-8"))

        if isinstance(data, dict) and "results" in data:
            items    = data["results"]
            wrap_key = "results"
        else:
            items    = data
            wrap_key = None

        # ── Load-time summary ─────────────────────────────────
        total_qas    = sum(len(item.get("qas", [])) for item in items)
        already_done = sum(
            1 for item in items for qa in item.get("qas", [])
            if isinstance(qa.get("llm_as_judge_score"), (int, float))
        )
        null_failed  = sum(
            1 for item in items for qa in item.get("qas", [])
            if "llm_as_judge_score" in qa and qa["llm_as_judge_score"] is None
        )
        not_yet_run  = total_qas - already_done - null_failed
        to_process   = null_failed + not_yet_run

        print(f"  Total QAs       : {total_qas}")
        print(f"  Already scored  : {already_done}  (will skip)")
        print(f"  Failed (null)   : {null_failed}   (will retry)")
        print(f"  Not yet run     : {not_yet_run}   (will run)")
        print(f"  To process      : {to_process}")
        print(f"{'─'*60}")
        # ─────────────────────────────────────────────────────

        # Full deep copy — we need all items so _save writes a complete file
        out_items = copy.deepcopy(items)

        done = failed = skipped = 0
        call_count = 0

        for item in out_items:
            image_id = item["id"]

            try:
                image_path = find_image(image_id, DATASET_FOLDER)
                img_b64, img_mime = encode_image(image_path)
                image_ok = True
            except FileNotFoundError as e:
                print(f"  [WARN] {e} — judging text-only for this item")
                image_ok = False

            for qa in item.get("qas", []):
                # Skip already-scored entries (non-null score)
                if "llm_as_judge_score" in qa and qa["llm_as_judge_score"] is not None:
                    skipped += 1
                    continue

                question     = qa.get("question", "")
                ground_truth = qa.get("ground_truth", "")
                predicted    = qa.get("answer", "")
                call_count  += 1

                print(f"  [{call_count}/{to_process}] qa_id={qa['qa_id']}  img={image_id}")

                prompt_text = build_prompt(question, ground_truth, predicted)

                try:
                    contents = (
                        [{"mime_type": img_mime, "data": img_b64}, prompt_text]
                        if image_ok else [prompt_text]
                    )
                    response = gemini.generate_content(
                        contents,
                        generation_config=genai.types.GenerationConfig(
                            temperature=0.0,
                            max_output_tokens=512,
                        ),
                    )
                    result = parse_score(response.text)

                except Exception as exc:
                    print(f"  [ERROR] {exc}")
                    result = {"score": None, "reason": f"api_error: {exc}"}

                qa["llm_as_judge_score"]  = result["score"]
                qa["llm_as_judge_reason"] = result["reason"]

                if result["score"] is not None:
                    done += 1
                else:
                    failed += 1

                # Persist after every QA — safe against interruptions
                _save(out_items, wrap_key, data, OUT_PATH)
                time.sleep(SLEEP_BETWEEN)

        _save(out_items, wrap_key, data, OUT_PATH)

        print(f"\nDone. Scored: {done}  Failed: {failed}  Skipped: {skipped}")
        print(f"Output → {OUT_PATH}")

        all_scores = [
            qa["llm_as_judge_score"]
            for item in out_items
            for qa in item.get("qas", [])
            if isinstance(qa.get("llm_as_judge_score"), (int, float))
        ]
        print_summary(all_scores)


if __name__ == "__main__":
    main()

# claude

In [ ]:
"""
LLM-as-a-Judge evaluation for Sinhala VQA using Claude (Anthropic).

Drop-in replacement for the Gemini judge — same scoring rubric, same
resume logic, same output format. Run alongside the Gemini judge and
average / compare the two scores for a dual-LLM evaluation.

Scores each predicted answer 0, 0.25, 0.5, or 1:
  1    — correct / semantically equivalent
  0.5  — partially correct (multi-part answer, got some parts)
  0.25 — right category/type, wrong specific value
  0    — wrong category entirely / random / nonsensical
"""

import os
import re
import json
import time
import base64
import copy
from pathlib import Path
from typing import Any

import anthropic

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MODEL_NAME     = "claude-sonnet-4-6"
DATASET_FOLDER = "../../benchmarking/data"
SLEEP_BETWEEN  = 0.3          # seconds between API calls
VALID_SCORES   = (0, 0.25, 0.5, 1)

# Output field names — distinct from Gemini so both judges can
# coexist in the same JSON file without overwriting each other.
SCORE_FIELD  = "claude_judge_score"
REASON_FIELD = "claude_judge_reason"

SYSTEM_PROMPT = (
    "You are a strict Visual Question Answering evaluator. "
    "You will be shown an image and a question about it written in Sinhala. "
    "Use the image to verify the ground truth and judge the predicted answer — "
    "do not rely solely on the ground truth text. "
    "Judge whether the predicted answer is correct using the ground truth as a semantic reference — "
    "not as the only valid wording. Treat semantically equivalent answers as correct "
    "(e.g. 'teddy', 'teddy bear', 'plushy toy' are all equivalent). "
    "Reply ONLY with a single JSON object: "
    "{\"score\": <0 or 0.25 or 0.5 or 1>, \"reason\": \"<one short sentence of max 15 words>\"}\n\n"
    "Score meanings:\n"
    "  1    — correct or semantically equivalent to the ground truth\n"
    "  0.5  — the ground truth requires multiple parts and the prediction covers some but not all "
    "(e.g. ground truth is 'red and blue', prediction is only 'blue')\n"
    "  0.25 — the answer type/category is correct but the specific value is wrong "
    "(e.g. question asks for a colour, ground truth is 'red', prediction is 'blue' — "
    "both are colours so the model understood what was asked, just got the wrong one; "
    "or question asks for a direction, ground truth is 'right', prediction is 'left')\n"
    "  0    — wrong category entirely, completely unrelated, or random/nonsensical text\n\n"
    "Important:\n"
    "- Use 0 when the prediction shows no understanding of what the question was asking for.\n"
    "- Use 0.25 when the model understood the question type but gave a wrong single value.\n"
    "- Use 0.5 when the answer is partially correct because the ground truth has multiple parts.\n"
    "- Use 1 only when the answer is fully correct or semantically equivalent."
)


# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────
def find_image(image_id: int, dataset_folder: str) -> str:
    extensions = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]
    for ext in extensions:
        image_path = os.path.join(dataset_folder, f"{image_id}{ext}")
        if os.path.exists(image_path):
            return image_path
        for root, _, _files in os.walk(dataset_folder):
            image_path = os.path.join(root, f"{image_id}{ext}")
            if os.path.exists(image_path):
                return image_path
    raise FileNotFoundError(f"Image {image_id} not found in {dataset_folder}")


def encode_image(image_path: str) -> tuple[str, str]:
    suffix = Path(image_path).suffix.lower()
    mime = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}.get(suffix, "image/jpeg")
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8"), mime


def build_user_content(
    question: str,
    ground_truth: str,
    predicted: str,
    img_b64: str | None,
    img_mime: str | None,
) -> list[dict]:
    """Build the user message content list for the Claude Messages API."""
    content: list[dict] = []

    if img_b64 and img_mime:
        content.append({
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": img_mime,
                "data": img_b64,
            },
        })

    content.append({
        "type": "text",
        "text": (
            f"Question (Sinhala): {question}\n"
            f"Ground truth answer: {ground_truth}\n"
            f"Predicted answer: {predicted}\n\n"
            "Score the predicted answer. Reply ONLY with JSON: "
            "{\"score\": <0 or 0.25 or 0.5 or 1>, \"reason\": \"...\"}"
        ),
    })

    return content


def parse_score(text: str) -> dict[str, Any]:
    """Extract score and reason — robust to truncated JSON."""
    text = (text or "").strip()

    # 1. Direct parse
    try:
        obj = json.loads(text)
        score = float(obj["score"])
        assert score in VALID_SCORES
        return {"score": score, "reason": obj.get("reason", "")}
    except Exception:
        pass

    # 2. Salvage first complete {...}
    l, r = text.find("{"), text.rfind("}")
    if 0 <= l < r:
        try:
            obj = json.loads(text[l : r + 1])
            score = float(obj["score"])
            assert score in VALID_SCORES
            return {"score": score, "reason": obj.get("reason", "")}
        except Exception:
            pass

    # 3. Regex fallback — works even if JSON is truncated mid-reason
    m = re.search(r'"score"\s*:\s*(0\.25|0\.5|0|1)', text)
    if m:
        score = float(m.group(1))
        r_match = re.search(r'"reason"\s*:\s*"([^"]*)"?', text)
        reason = r_match.group(1) if r_match else "(reason truncated)"
        return {"score": score, "reason": reason}

    return {"score": None, "reason": f"parse_failed: {text[:120]}"}


def print_summary(all_scores: list[float]) -> None:
    if not all_scores:
        return
    avg = sum(all_scores) / len(all_scores)
    counts = {0: 0, 0.25: 0, 0.5: 0, 1: 0}
    for s in all_scores:
        if s in counts:
            counts[s] += 1
    n = len(all_scores)
    print(f"\nScore distribution over {n} QAs:")
    print(f"  1    (correct)             : {counts[1]:5d}  ({100*counts[1]/n:.1f}%)")
    print(f"  0.5  (partial multi-part)  : {counts[0.5]:5d}  ({100*counts[0.5]/n:.1f}%)")
    print(f"  0.25 (right type/wrong val): {counts[0.25]:5d}  ({100*counts[0.25]/n:.1f}%)")
    print(f"  0    (wrong)               : {counts[0]:5d}  ({100*counts[0]/n:.1f}%)")
    print(f"  Average score              : {avg:.4f}")


def _save(out_items, wrap_key, original_data, out_path: Path):
    if wrap_key:
        out = {**original_data, wrap_key: out_items}
    else:
        out = out_items
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    api_key = API_KEY
    if not api_key:
        raise RuntimeError("Export ANTHROPIC_API_KEY before running.")

    client = anthropic.Anthropic(api_key=api_key)

    files = [
        "smolvlm-500m.json",
       
        
    ]
    SKIP = {"scores.json"}

    for file in files:
        if file in SKIP:
            continue

        IN_PATH  = Path(f"../2_CLAUDE/{file}")
        # Write Claude scores to a parallel file so Gemini scores are untouched
        OUT_PATH = Path(f"{file}")

        print(f"\n{'='*60}")
        print(f"Processing: {file}")
        print(f"{'='*60}")

        # Load from the Gemini-scored file if it exists, else from raw
        src_path = OUT_PATH if OUT_PATH.exists() else IN_PATH
        data = json.loads(src_path.read_text(encoding="utf-8"))

        if isinstance(data, dict) and "results" in data:
            items    = data["results"]
            wrap_key = "results"
        else:
            items    = data
            wrap_key = None

        # ── Load-time summary ─────────────────────────────────
        total_qas    = sum(len(item.get("qas", [])) for item in items)
        already_done = sum(
            1 for item in items for qa in item.get("qas", [])
            if isinstance(qa.get(SCORE_FIELD), (int, float))
        )
        null_failed  = sum(
            1 for item in items for qa in item.get("qas", [])
            if SCORE_FIELD in qa and qa[SCORE_FIELD] is None
        )
        not_yet_run  = total_qas - already_done - null_failed
        to_process   = null_failed + not_yet_run

        print(f"  Total QAs       : {total_qas}")
        print(f"  Already scored  : {already_done}  (will skip)")
        print(f"  Failed (null)   : {null_failed}   (will retry)")
        print(f"  Not yet run     : {not_yet_run}   (will run)")
        print(f"  To process      : {to_process}")
        print(f"{'─'*60}")

        out_items = copy.deepcopy(items)
        done = failed = skipped = 0
        call_count = 0

        for item in out_items:
            image_id = item["id"]

            try:
                image_path = find_image(image_id, DATASET_FOLDER)
                img_b64, img_mime = encode_image(image_path)
                image_ok = True
            except FileNotFoundError as e:
                print(f"  [WARN] {e} — judging text-only for this item")
                img_b64, img_mime = None, None
                image_ok = False

            for qa in item.get("qas", []):
                # Resume: skip already-scored entries (non-null score)
                if SCORE_FIELD in qa and qa[SCORE_FIELD] is not None:
                    skipped += 1
                    continue

                question     = qa.get("question", "")
                ground_truth = qa.get("ground_truth", "")
                predicted    = qa.get("answer", "")
                call_count  += 1

                print(f"  [{call_count}/{to_process}] qa_id={qa['qa_id']}  img={image_id}")

                user_content = build_user_content(
                    question, ground_truth, predicted,
                    img_b64 if image_ok else None,
                    img_mime if image_ok else None,
                )

                try:
                    response = client.messages.create(
                        model=MODEL_NAME,
                        max_tokens=256,
                        system=SYSTEM_PROMPT,
                        messages=[{"role": "user", "content": user_content}],
                    )
                    raw_text = response.content[0].text
                    result = parse_score(raw_text)

                except anthropic.RateLimitError as exc:
                    # Back off and retry once on rate limit
                    print(f"  [RATE LIMIT] backing off 60 s… ({exc})")
                    time.sleep(60)
                    try:
                        response = client.messages.create(
                            model=MODEL_NAME,
                            max_tokens=256,
                            system=SYSTEM_PROMPT,
                            messages=[{"role": "user", "content": user_content}],
                        )
                        raw_text = response.content[0].text
                        result = parse_score(raw_text)
                    except Exception as exc2:
                        result = {"score": None, "reason": f"api_error: {exc2}"}

                except Exception as exc:
                    print(f"  [ERROR] {exc}")
                    result = {"score": None, "reason": f"api_error: {exc}"}

                qa[SCORE_FIELD]  = result["score"]
                qa[REASON_FIELD] = result["reason"]

                if result["score"] is not None:
                    done += 1
                else:
                    failed += 1

                _save(out_items, wrap_key, data, OUT_PATH)
                time.sleep(SLEEP_BETWEEN)

        _save(out_items, wrap_key, data, OUT_PATH)

        print(f"\nDone. Scored: {done}  Failed: {failed}  Skipped: {skipped}")
        print(f"Output → {OUT_PATH}")

        all_scores = [
            qa[SCORE_FIELD]
            for item in out_items
            for qa in item.get("qas", [])
            if isinstance(qa.get(SCORE_FIELD), (int, float))
        ]
        print_summary(all_scores)


if __name__ == "__main__":
    main()